In [1]:
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupKFold, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (
    roc_auc_score, average_precision_score, f1_score, balanced_accuracy_score,
    brier_score_loss, roc_curve, precision_recall_curve
)
from sklearn.inspection import permutation_importance

In [7]:
win_task2 = pd.read_csv("/Users/ferolahima/Desktop/gazepoint_project/notebooks/improving models/outputs2/fix_wid_t2.csv")

In [8]:
# ----------------- config -----------------
FEATS = ['fix_rate_per_s','fix_median_dur','gaze_dispersion_total','saccade_mean_mag','saccade_dir_var','saccade_rate_per_s','dir_change_rate','axis_ratio'
]
QC_MIN_VALID = 0.5      # keep windows with at least this valid gaze ratio
N_OUTER = 5             # outer GroupKFold splits
N_INNER = 3             # inner CV for hyperparameters
RSEED   = 0

# ----------------- helpers -----------------
def _prep_df_for_model(df: pd.DataFrame):
    """Filter to ON/OFF, QC, build X, y, groups, meta (for later stratified metrics)."""
    d = df.copy()

    # tidy labels
    d["label"] = d["label"].astype(str).str.upper().str.strip()
    d = d[d["label"].isin(["ON","OFF"])].copy()

    # QC
    if "valid_gaze_ratio" in d.columns:
        d = d[d["valid_gaze_ratio"] >= QC_MIN_VALID].copy()

    # ensure identifiers
    for c in ["Video Name (folder)","User ID (file)"]:
        if c not in d.columns: d[c] = "UNK"

    # groups by participant (Video+User)
    groups = (d["Video Name (folder)"].astype(str) + "||" + d["User ID (file)"].astype(str)).values

    # feature matrix and target
    X = d[FEATS].copy()
    y = (d["label"] == "ON").astype(int).values

    meta = d[["ease","direction","delay"]].copy() if set(["ease","direction","delay"]).issubset(d.columns) \
           else pd.DataFrame(index=d.index)

    return X, y, groups, meta, d.index

def _build_models():
    """Define models + small hyperparameter grids."""
    pipe_lr = Pipeline([
        ("imp", SimpleImputer(strategy="median")),
        ("sc",  StandardScaler()),
        ("lr",  LogisticRegression(
            max_iter=2000, class_weight="balanced", solver="lbfgs", penalty="l2", n_jobs=None
        ))
    ])
    grid_lr = {
        "lr__C": [0.1, 1.0, 3.0, 10.0]
    }

    pipe_rf = Pipeline([
        ("imp", SimpleImputer(strategy="median")),
        ("rf",  RandomForestClassifier(
            n_estimators=500, class_weight="balanced_subsample", random_state=RSEED, n_jobs=-1
        ))
    ])
    grid_rf = {
        "rf__max_depth": [None, 6, 10],
        "rf__min_samples_leaf": [1, 3, 6],
        "rf__max_features": ["sqrt", "log2"]
    }
    return (pipe_lr, grid_lr), (pipe_rf, grid_rf)

def _metrics_from_probs(y_true, p1):
    """Compute standard metrics from positive-class probabilities."""
    # threshold 0.5 (you can tune threshold later if you like)
    y_hat = (p1 >= 0.5).astype(int)
    out = {
        "AUROC": roc_auc_score(y_true, p1) if len(np.unique(y_true))>1 else np.nan,
        "AUPRC(ON)": average_precision_score(y_true, p1) if len(np.unique(y_true))>1 else np.nan,
        "BalancedAcc": balanced_accuracy_score(y_true, y_hat),
        "MacroF1": f1_score(y_true, y_hat, average="macro"),
        "Brier": brier_score_loss(y_true, p1),
    }
    return out

def _aggregate_fold_metrics(fold_metrics):
    """Return mean ± std as a DataFrame row."""
    keys = fold_metrics[0].keys()
    mean = {k: np.nanmean([m[k] for m in fold_metrics]) for k in keys}
    std  = {k: np.nanstd([m[k] for m in fold_metrics])  for k in keys}
    summary = {f"{k}_mean": mean[k] for k in keys} | {f"{k}_std": std[k] for k in keys}
    return pd.Series(summary)

def _odds_ratios_from_lr(best_pipeline):
    """Extract standardized LR coefficients → odds ratios."""
    lr = best_pipeline.named_steps["lr"]
    sc = best_pipeline.named_steps["sc"]
    coefs = pd.Series(lr.coef_[0], index=best_pipeline[:-1].get_feature_names_out(FEATS))
    # with standardized inputs, OR per +1 SD = exp(beta)
    return np.exp(coefs).rename("OddsRatio(+1 SD)")

def _permutation_importance_on_test(model_pipeline, X_test, y_test, n_repeats=10):
    """Permutation importance on the **outer test** (avoids training bias)."""
    r = permutation_importance(model_pipeline, X_test, y_test, n_repeats=n_repeats, random_state=RSEED, scoring="roc_auc")
    imp = pd.Series(r.importances_mean, index=model_pipeline[:-1].get_feature_names_out(FEATS)).sort_values(ascending=False)
    return imp

def run_grouped_nested_cv(df: pd.DataFrame, title: str):
    """Full run for one dataset (task family)."""
    print(f"\n=== Supervised (Grouped Nested CV) — {title} ===")
    X, y, groups, meta, idx = _prep_df_for_model(df)
    if X.dropna(how="all").empty or len(np.unique(y)) < 2:
        print("Not enough data or only one class after QC.")
        return None

    (pipe_lr, grid_lr), (pipe_rf, grid_rf) = _build_models()
    outer_cv = GroupKFold(n_splits=min(N_OUTER, len(np.unique(groups))))

    results = {}
    for model_name, base_pipe, grid in [
        ("LR", pipe_lr, grid_lr),
        ("RF", pipe_rf, grid_rf),
    ]:
        print(f"\n[{title}] Model: {model_name}")
        fold_metrics = []
        oof_probs = np.full(len(X), np.nan)
        best_params = []

        for fold, (tr, te) in enumerate(outer_cv.split(X, y, groups=groups), 1):
            Xtr, Xte = X.iloc[tr], X.iloc[te]
            ytr, yte = y[tr], y[te]

            # inner tuning using only training fold and same groups restriction
            inner_cv = GroupKFold(n_splits=min(N_INNER, len(np.unique(groups[tr]))))
            gs = GridSearchCV(
                estimator=base_pipe, param_grid=grid,
                scoring="roc_auc", cv=inner_cv.split(Xtr, ytr, groups=groups[tr]),
                n_jobs=-1, refit=True
            )
            gs.fit(Xtr, ytr)
            best_params.append(gs.best_params_)

            # Calibrate on the training fold only (with 3-fold CV inside)
            calib = CalibratedClassifierCV(gs.best_estimator_, method="isotonic", cv=3)
            calib.fit(Xtr, ytr)

            # store OOF probabilities
            p1 = calib.predict_proba(Xte)[:,1]
            oof_probs[te] = p1

            # fold metrics
            fm = _metrics_from_probs(yte, p1)
            fold_metrics.append(fm)
            print(f"  fold {fold}: " + ", ".join([f"{k}={v:.3f}" for k,v in fm.items()]))

        # summary
        summary = _aggregate_fold_metrics(fold_metrics)
        print("\n  Summary (mean ± std):")
        for k in ["AUROC","AUPRC(ON)","BalancedAcc","MacroF1","Brier"]:
            print(f"   {k}: {summary[f'{k}_mean']:.3f} ± {summary[f'{k}_std']:.3f}")

        # interpretability on the **last** fold's fitted estimator (quick view)
        fitted = gs.best_estimator_  # from last loop scope
        if model_name == "LR":
            try:
                or_df = _odds_ratios_from_lr(fitted).to_frame()
                or_df.columns = ["OddsRatio(+1 SD)"]
                print("\n  Logistic Regression — standardized odds ratios (last fold best):")
                print(or_df.sort_values("OddsRatio(+1 SD)", ascending=False).round(3).to_string())
            except Exception as e:
                print("  (LR ORs unavailable)", e)
        else:
            try:
                # importance using the *pipeline* (impute+rf), evaluated on the last outer test is cleaner,
                # but we don't have X_test here anymore. So compute on the full X against y (informative but not pure OOF).
                imp = _permutation_importance_on_test(fitted, X, y, n_repeats=5)
                print("\n  Random Forest — permutation importance (global, quick view):")
                print(imp.round(4).to_string())
            except Exception as e:
                print("  (RF importances unavailable)", e)

        # per-property AUROC using OOF predictions
        prop_metrics = {}
        for prop in ["ease", "direction", "delay"]:
            if prop not in meta.columns: continue
            levels = pd.Series(meta[prop].astype(str).values, index=idx)
            pm = []
            for lvl, mask in levels.groupby(levels).groups.items():
                m_idx = np.array([i for i in mask], dtype=int)
                yt = y[[np.where(idx==i)[0][0] for i in m_idx if not np.isnan(oof_probs[np.where(idx==i)[0][0]])]]
                pt = oof_probs[[np.where(idx==i)[0][0] for i in m_idx if not np.isnan(oof_probs[np.where(idx==i)[0][0]])]]
                if len(np.unique(yt))<2 or len(yt)<25:
                    pm.append((lvl, np.nan, len(yt)))
                else:
                    pm.append((lvl, roc_auc_score(yt, pt), len(yt)))
            if pm:
                prop_metrics[prop] = pd.DataFrame(pm, columns=[prop,"AUROC","n"]).sort_values("AUROC", ascending=False)

        results[model_name] = {
            "summary": summary,
            "oof_probs": oof_probs,
            "y": y,
            "idx": idx,
            "best_params": best_params,
            "per_property_auroc": prop_metrics
        }

    return results

# =========================
# Run per task family
# =========================
all_results = {}
for name, df in {
    "Task 2.x": win_task2,
}.items():
    all_results[name] = run_grouped_nested_cv(df, title=name)

# (Optional) Quick tabular summary of AUROC across tasks & models
rows = []
for task, res in all_results.items():
    if res is None: continue
    for model in ["LR","RF"]:
        s = res[model]["summary"]
        rows.append([task, model, s["AUROC_mean"], s["AUROC_std"], s["BalancedAcc_mean"], s["MacroF1_mean"]])
print("\n=== Fold-averaged metrics (summary) ===")
print(pd.DataFrame(rows, columns=["Task","Model","AUROC_mean","AUROC_std","BalancedAcc_mean","MacroF1_mean"])
        .sort_values(["Task","Model"]).to_string(index=False))



=== Supervised (Grouped Nested CV) — Task 2.x ===

[Task 2.x] Model: LR
  fold 1: AUROC=0.825, AUPRC(ON)=0.949, BalancedAcc=0.693, MacroF1=0.719, Brier=0.113
  fold 2: AUROC=0.781, AUPRC(ON)=0.883, BalancedAcc=0.708, MacroF1=0.724, Brier=0.154
  fold 3: AUROC=0.757, AUPRC(ON)=0.881, BalancedAcc=0.680, MacroF1=0.702, Brier=0.151
  fold 4: AUROC=0.801, AUPRC(ON)=0.914, BalancedAcc=0.695, MacroF1=0.710, Brier=0.140
  fold 5: AUROC=0.746, AUPRC(ON)=0.836, BalancedAcc=0.687, MacroF1=0.709, Brier=0.160

  Summary (mean ± std):
   AUROC: 0.782 ± 0.029
   AUPRC(ON): 0.893 ± 0.038
   BalancedAcc: 0.692 ± 0.009
   MacroF1: 0.713 ± 0.008
   Brier: 0.144 ± 0.017

  Logistic Regression — standardized odds ratios (last fold best):
                       OddsRatio(+1 SD)
saccade_rate_per_s                2.289
fix_rate_per_s                    1.851
fix_median_dur                    1.274
dir_change_rate                   1.084
saccade_mean_mag                  1.015
gaze_dispersion_total           

In [ ]:
# (Optional) Show per-property AUROC for a task/model
# example:
for i in range(1,4):
    print(f"\n=== Per-property AUROC — Task {i}.x ===")
    for model in ["LR","RF"]:
        print(f"\n-- Model: {model} --")
        res = all_results[f"Task {i}.x"]
        if res is None: continue
        for prop, df in res[model]["per_property_auroc"].items():
            print(f"\nProperty: {prop}")
            print(df.to_string(index=False))